# 01 — Complete ICFD-31k Dataset Exploration, Audit & Processing
**Project:** Real-Time Fraud Detection Pipeline (Simulated Streaming Anomaly Detection)  
**Dataset:** Indian Conversational Fraud Dataset (`rishia2220/icfd-31k`, v1.0) — *synthetic, human-in-the-loop validated English/Hinglish conversations*

### End-to-End Steps Covered in This Notebook
1. **Environment & Path Setup:** Configures local folders (`../data/icfd-31k`, `../data/processed`, `../outputs/eda`).
2. **Remote Size Check & Idempotent Download:** Verifies Hugging Face repo size (~61.66 MB) and downloads only if missing.
3. **Official Metadata & Single-Shard Schema Check:** Reads `release_summary.json` and inspects `train-00023-of-00050.parquet` via `pyarrow.parquet` (bypassing the Python 3.14 `pandas`/`pyarrow` `ArrowKeyError`).
4. **Shard Boundary Demonstration:** Proves why single-shard loading splits calls across file borders.
5. **Full Split Stitching & Call Indexing:** Loads all 68 shards across `train`, `validation`, `test`, and `cross_domain`, stitches split calls back together, sorts chronologically by `["conversation_uid", "chunk_timestamp"]`, builds lightweight `call_map` index arrays, and measures exact RAM usage.
6. **Data Integrity & Leakage Audit:** Verifies 0 session overlap across splits, timestamp step intervals (3s), verdict consistency within calls, missing values, and explicit leakage vs. safe model feature columns.
7. **Complete Call-Level & Chunk-Level EDA (For Slide 6):** Split balance, 10 main domains, `case_type` breakdown, call duration & word-count growth, and a side-by-side Fraud vs. Legitimate call progression sample.
8. **Audio Metadata Parsing (`cumulative_audio_analysis_json`):** Extracts `dominant_emotion`, `secondary_emotion`, `pace`, and `confidence_score` and compares Fraud vs. Legitimate distributions.
9. **Pause / Repeated-Text Analysis:** Quantifies consecutive identical-text chunks (trailing/mid-call silence).
10. **Presentation-Ready Charts & Processed Exports:** Saves 5 high-resolution PNG charts + `eda_summary_stats.json` to `../outputs/eda/` and writes stitched, leakage-tagged Parquet files to `../data/processed/` via `pyarrow` for Notebook 02.

In [ ]:
# If matplotlib is not installed in your environment yet, uncomment and run once:
# %pip install -q matplotlib

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import HfApi, snapshot_download

# Project paths (notebook lives in notebooks/)
data_dir = Path("../data/icfd-31k")
processed_dir = Path("../data/processed")
eda_out_dir = Path("../outputs/eda")

processed_dir.mkdir(parents=True, exist_ok=True)
eda_out_dir.mkdir(parents=True, exist_ok=True)

# Clean plotting defaults for Slide 6 charts
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print("Raw dataset dir :", data_dir.resolve())
print("Processed dir   :", processed_dir.resolve())
print("EDA output dir  :", eda_out_dir.resolve())

## Step 1: Check Repository Size on Hugging Face
Inspects remote file sizes before downloading. Skips gracefully if you are offline.

In [ ]:
try:
    api = HfApi()
    repo_info = api.dataset_info("rishia2220/icfd-31k", files_metadata=True)

    total_bytes = 0
    print(f"{'File Path':<58} | {'Size (MB)':>10}")
    print("-" * 73)

    for file in repo_info.siblings:
        size_bytes = file.size or 0
        total_bytes += size_bytes
        size_mb = size_bytes / (1024 ** 2)
        print(f"{file.rfilename:<58} | {size_mb:>10.2f} MB")

    print("-" * 73)
    print(
        f"Total Repository Size: {total_bytes / (1024 ** 3):.2f} GB "
        f"({total_bytes / (1024 ** 2):.2f} MB)"
    )
except Exception as exc:
    print(f"Skipping remote size check (offline or HF Hub unavailable): {exc}")

## Step 2: Idempotent Local Download & Official Metadata (`release_summary.json`)
Downloads the dataset once to `../data/icfd-31k` and skips downloading on future runs.

In [ ]:
summary_file = data_dir / "release_summary.json"

if not summary_file.exists():
    print("Dataset not found locally. Downloading from Hugging Face...")
    snapshot_download(
        repo_id="rishia2220/icfd-31k",
        repo_type="dataset",
        local_dir=str(data_dir),
    )
    print(f"Dataset downloaded to: {data_dir.resolve()}")
else:
    print(f"Dataset already exists at '{data_dir.resolve()}'")

with open(summary_file, "r", encoding="utf-8") as f:
    release_summary = json.load(f)

print(json.dumps(release_summary, indent=2))

## Step 3: Single-Shard Schema Verification (`pyarrow.parquet`)
Reads `train-00023-of-00050.parquet` directly through `pyarrow.parquet` (avoiding the Python 3.14 `ArrowKeyError`) and compares Row 0 vs. Row 1.

In [ ]:
sample_file = data_dir / "streaming_chunks" / "train-00023-of-00050.parquet"
df_sample = pq.read_table(sample_file).to_pandas()

print("Rows in this shard:", len(df_sample))
print("\nColumns & Data Types:")
print(df_sample.dtypes)

print("\n--- First Row Values ---")
for col, val in df_sample.iloc[0].items():
    preview = str(val)
    if len(preview) > 220:
        preview = preview[:220] + " ..."
    print(f"  {col}: {preview}")

print("\n--- Columns That Differ Between Row 0 and Row 1 ---")
for col in df_sample.columns:
    v0, v1 = df_sample.iloc[0][col], df_sample.iloc[1][col]
    if pd.isna(v0) and pd.isna(v1):
        continue
    if v0 != v1:
        print(f"  {col}: {v0!r}  -->  {v1!r}")

## Step 4: Single-Shard Grouping Check (Why Stitching All Shards Is Required)
Sorting and grouping a single shard by `conversation_uid` shows that calls near file boundaries are split across consecutive shards (e.g., Call 0 only has `[96, 99]` in shard 23).

In [ ]:
df_sorted_sample = df_sample.sort_values(["conversation_uid", "chunk_timestamp"]).reset_index(drop=True)

sample_call_map = {
    uid: group.index.to_numpy()
    for uid, group in df_sorted_sample.groupby("conversation_uid", sort=False)
}
sample_calls_list = list(sample_call_map.values())

print(f"Total unique calls in shard 23: {len(sample_calls_list)}")
for idx in range(3):
    ts = df_sorted_sample.loc[sample_calls_list[idx], "chunk_timestamp"].tolist()
    preview = f"{ts[:6]} ... {ts[-2:]}" if len(ts) > 8 else str(ts)
    print(f"  Call {idx} ({len(ts):>2} chunks): {preview}")

## Step 5: Load All 68 Shards Per Split, Stitch Split Calls & Build Call Index Maps
`load_and_group_split()` reads all Parquet shards for a split in a single `pq.read_table()` call so every call cut across shard borders is reunited, sorted chronologically (`chunk_timestamp`), and indexed in `call_map` (`{conversation_uid: np.ndarray of row indices}`).

In [ ]:
def load_and_group_split(split_name):
    """Load all parquet shards for a split, sort each call chronologically,
    and map each conversation_uid to its sorted row indices."""
    shard_files = sorted((data_dir / "streaming_chunks").glob(f"{split_name}-*.parquet"))
    if not shard_files:
        raise FileNotFoundError(f"No parquet shards found for split '{split_name}'")

    # Read all shards together so calls split across shard borders are reunited
    df = pq.read_table(shard_files).to_pandas()
    df = df.sort_values(["conversation_uid", "chunk_timestamp"]).reset_index(drop=True)

    # Lightweight reference map: { conversation_uid: np.ndarray of row indices }
    call_map = {
        uid: group.index.to_numpy()
        for uid, group in df.groupby("conversation_uid", sort=False)
    }
    return df, call_map


train_df, train_calls = load_and_group_split("train")
val_df, val_calls     = load_and_group_split("validation")
test_df, test_calls   = load_and_group_split("test")
cross_df, cross_calls = load_and_group_split("cross_domain")

splits = {
    "train": (train_df, train_calls),
    "validation": (val_df, val_calls),
    "test": (test_df, test_calls),
    "cross_domain": (cross_df, cross_calls),
}

print(f"{'Split':<14} | {'Calls':>8} | {'Chunks':>10} | {'Avg Chunks/Call':>15} | {'RAM (MB)':>10}")
print("-" * 68)
total_calls = 0
total_chunks = 0
total_ram = 0.0
for name, (df, cmap) in splits.items():
    ram_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    total_calls += len(cmap)
    total_chunks += len(df)
    total_ram += ram_mb
    avg_chunks = len(df) / len(cmap)
    print(f"{name:<14} | {len(cmap):>8,} | {len(df):>10,} | {avg_chunks:>15.2f} | {ram_mb:>10.1f}")
print("-" * 68)
print(f"{'TOTAL':<14} | {total_calls:>8,} | {total_chunks:>10,} | {total_chunks / total_calls:>15.2f} | {total_ram:>10.1f}")

## Step 6: Data Integrity, Streaming Cadence & Data Leakage Audit
Verifies:
1. **Zero Session Overlap:** No `conversation_uid` leaks across `train`, `validation`, `test`, or `cross_domain`.
2. **Streaming Step Size:** Confirms exact time gap (`chunk_timestamp` diff) between consecutive chunks within calls.
3. **Verdict Stability:** Checks whether `final_verdict` stays constant or changes mid-call.
4. **Missing Values:** Flags every column with nulls.
5. **Leakage Prevention Policy:** Defines exact `LEAKAGE_COLS` vs. `SAFE_MODEL_INPUT_COLS` for Notebook 02.

In [ ]:
# 1. Cross-split session overlap check
split_names = list(splits.keys())
print("=== 1. Cross-Split Session Overlap Check ===")
overlap_found = False
for i in range(len(split_names)):
    for j in range(i + 1, len(split_names)):
        s1, s2 = split_names[i], split_names[j]
        shared = set(splits[s1][1].keys()) & set(splits[s2][1].keys())
        if shared:
            overlap_found = True
            print(f"  WARNING: {len(shared)} overlapping calls between {s1} and {s2}!")
if not overlap_found:
    print("  PASS: 0 overlapping conversation_uid values across all 4 splits.")

# 2. Verify 3-second chunk cadence within calls (Train)
print("\n=== 2. Consecutive Chunk Timestamp Step Sizes (Train) ===")
same_call_mask = train_df["conversation_uid"] == train_df["conversation_uid"].shift(1)
ts_diffs = train_df.loc[same_call_mask, "chunk_timestamp"] - train_df["chunk_timestamp"].shift(1)[same_call_mask]
print(ts_diffs.value_counts().sort_index().head(10))

# 3. Check if final_verdict ever changes mid-call (Train)
print("\n=== 3. Verdict Stability Within Calls (Train) ===")
verdicts_per_call = train_df.groupby("conversation_uid")["final_verdict"].nunique()
print("Distinct final_verdict values per call:")
print(verdicts_per_call.value_counts().sort_index())

# 4. Missing values in Train
print("\n=== 4. Missing Values in Train Split ===")
missing = train_df.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print("  No missing values found.")
else:
    print(missing)

# 5. Explicit Leakage vs. Safe Model Feature Columns
LEAKAGE_COLS = [
    "final_verdict",             # Ground-truth label (evaluation & threshold calibration ONLY)
    "slow_thinking_rationale",   # Policy explanation revealing the answer
    "case_type",                 # Explicitly labels Clear Normal / fraud type
    "scenario_info_json",        # Contains case_type and scenario description
    "source_file_stem",          # Filename embeds Clear_Normal / fraud info
    "chunk_file",                # Filename embeds source_file_stem
    "conversation_uid",          # Embeds source_file_stem (use ONLY for grouping/eval)
    "scenario_id",               # Maps directly to scenario label definitions
]

SAFE_MODEL_INPUT_COLS = [
    "cumulative_text",                  # -> Sentence embeddings
    "cumulative_audio_analysis_json",   # -> Encoded emotion, pace, confidence_score
    "chunk_timestamp",                  # -> Normalized call time progression
]

print("\n=== 5. Modeling Column Policy (For Notebook 02) ===")
print("  DROP (Leakage / Eval Only):", LEAKAGE_COLS)
print("  KEEP (Feature Extraction) :", SAFE_MODEL_INPUT_COLS)

## Step 7: Call-Level & Chunk-Level EDA Statistics (For Slide 6)
Aggregates chronologically sorted chunks to **1 row per phone call** so long calls do not skew call-level percentages, then computes:
* Call-level class balance across all 4 splits
* Chunk-level class balance in `train`
* Call duration, chunk count, and transcript length distributions
* Domain breakdown across all 10 fraud domains + `case_type` breakdown
* Side-by-side sample of a **Fraudulent** vs. **Legitimate** call unfolding over time

In [ ]:
def build_call_summary(df):
    """Aggregate chronologically sorted streaming chunks to 1 row per call."""
    summary = (
        df.groupby("conversation_uid", sort=False)
        .agg(
            domain=("domain", "first"),
            case_type=("case_type", "first"),
            final_verdict=("final_verdict", "last"),
            num_chunks=("chunk_timestamp", "size"),
            first_ts=("chunk_timestamp", "min"),
            last_ts=("chunk_timestamp", "max"),
            last_text=("cumulative_text", "last"),
        )
        .reset_index()
    )
    last_text_str = summary["last_text"].fillna("").astype(str)
    summary["final_text_chars"] = last_text_str.str.len()
    summary["final_text_words"] = last_text_str.str.split().str.len()
    return summary.drop(columns=["last_text"])


call_summaries = {name: build_call_summary(df) for name, (df, _) in splits.items()}
train_calls_df = call_summaries["train"]

# 1. Call-level Final Verdict across all splits
print("=== 1. Call-Level Final Verdict Distribution Across Splits ===")
verdict_table = pd.DataFrame(
    {name: cs["final_verdict"].value_counts() for name, cs in call_summaries.items()}
).fillna(0).astype(int)
verdict_table["Total"] = verdict_table.sum(axis=1)
print(verdict_table)

# 2. Chunk-level Final Verdict in Train
print("\n=== 2. Chunk-Level Final Verdict Distribution (Train) ===")
chunk_verdicts = train_df["final_verdict"].value_counts(dropna=False)
chunk_pcts = (chunk_verdicts / len(train_df) * 100).round(2)
print(pd.DataFrame({"chunks": chunk_verdicts, "pct_%": chunk_pcts}))

# 3. Call Duration & Transcript Length Summary (Train)
print("\n=== 3. Call Duration, Chunk Count & Word Count Summary (Train) ===")
print(
    train_calls_df[["num_chunks", "first_ts", "last_ts", "final_text_chars", "final_text_words"]]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95])
    .round(2)
)

# 4. Domain Breakdown (Train — Call Level)
print("\n=== 4. Domain Breakdown (Train — Call Level) ===")
domain_ct = pd.crosstab(train_calls_df["domain"], train_calls_df["final_verdict"])
domain_ct["Total"] = domain_ct.sum(axis=1)
if "YES" in domain_ct.columns:
    domain_ct["Fraud_%"] = (domain_ct["YES"] / domain_ct["Total"] * 100).round(1)
print(domain_ct.sort_values("Total", ascending=False))

# 5. Case Type Breakdown (Train — Call Level)
print("\n=== 5. Case Type Breakdown (Train — Call Level) ===")
case_type_ct = pd.crosstab(train_calls_df["case_type"], train_calls_df["final_verdict"], margins=True)
print(case_type_ct)

In [ ]:
# Side-by-side preview: 1 Legitimate call vs. 1 Fraudulent call unfolding across early/mid/end chunks
for target_verdict in sorted(train_calls_df["final_verdict"].dropna().unique()):
    matching = train_calls_df[train_calls_df["final_verdict"] == target_verdict]
    if matching.empty:
        continue
    sample_uid = matching.iloc[0]["conversation_uid"]
    rows_idx = train_calls[sample_uid]
    sub = train_df.loc[rows_idx]
    pick_positions = sorted(set([0, len(sub) // 2, len(sub) - 1]))

    print("=" * 80)
    print(f"SAMPLE CALL — Verdict: {target_verdict} | Domain: {sub.iloc[0]['domain']} | Case Type: {sub.iloc[0]['case_type']}")
    print(f"UID: {sample_uid} | Total Chunks: {len(sub)} | Duration: {sub.iloc[0]['chunk_timestamp']}s -> {sub.iloc[-1]['chunk_timestamp']}s")
    print("=" * 80)
    for pos in pick_positions:
        r = sub.iloc[pos]
        txt = str(r["cumulative_text"]).replace("\n", " ")
        if len(txt) > 260:
            txt = txt[:260] + " ..."
        print(f"  [t={r['chunk_timestamp']:>3}s | verdict={r['final_verdict']}] audio={r['cumulative_audio_analysis_json']}")
        print(f"    text: {txt}")

## Step 8: Parse `cumulative_audio_analysis_json` (Emotion, Pace, Confidence)
Safely parses the structured JSON string in `cumulative_audio_analysis_json` across all training chunks to inspect unique emotions, speech pace, and confidence scores—and compares Fraud vs. Legitimate chunks.

In [ ]:
def safe_parse_json(val):
    if isinstance(val, dict):
        return val
    if pd.isna(val):
        return {}
    text = str(val).strip()
    if not text:
        return {}
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {}


audio_records = [safe_parse_json(x) for x in train_df["cumulative_audio_analysis_json"]]
audio_df = pd.DataFrame(audio_records)
audio_df["final_verdict"] = train_df["final_verdict"].values

audio_feature_cols = [c for c in audio_df.columns if c != "final_verdict"]
print("Parsed audio fields:", audio_feature_cols)
print("Rows with empty/missing audio JSON:", audio_df[audio_feature_cols].isna().all(axis=1).sum())

if "dominant_emotion" in audio_df.columns:
    print("\n=== Top 12 Dominant Emotions (Train Chunks) ===")
    print(audio_df["dominant_emotion"].value_counts(dropna=False).head(12))

if "secondary_emotion" in audio_df.columns:
    print("\n=== Top 10 Secondary Emotions (Train Chunks) ===")
    print(audio_df["secondary_emotion"].value_counts(dropna=False).head(10))

if "pace" in audio_df.columns:
    print("\n=== Speech Pace Distribution (Train Chunks) ===")
    print(audio_df["pace"].value_counts(dropna=False))

    print("\n=== Speech Pace (%) by Final Verdict ===")
    pace_by_verdict = (
        pd.crosstab(audio_df["pace"].fillna("Missing"), audio_df["final_verdict"], normalize="columns")
        * 100
    )
    print(pace_by_verdict.round(2))
else:
    pace_by_verdict = pd.DataFrame()

if "confidence_score" in audio_df.columns:
    print("\n=== Confidence Score Summary ===")
    print(pd.to_numeric(audio_df["confidence_score"], errors="coerce").describe().round(4))

## Step 9: Quantifying Pauses / Repeated Consecutive Chunks
Measures how many consecutive 3-second chunks within the same call have identical `cumulative_text` (e.g., pauses between utterances or trailing silence before hang-up).

In [ ]:
same_call = train_df["conversation_uid"] == train_df["conversation_uid"].shift(1)
same_text = train_df["cumulative_text"] == train_df["cumulative_text"].shift(1)
repeated_chunks = int((same_call & same_text).sum())
unique_prog_chunks = len(train_df) - repeated_chunks

print(f"Total train chunks                 : {len(train_df):,}")
print(f"Consecutive identical-text chunks  : {repeated_chunks:,} ({repeated_chunks / len(train_df) * 100:.2f}%)")
print(f"Unique-progression chunks          : {unique_prog_chunks:,} ({unique_prog_chunks / len(train_df) * 100:.2f}%)")

## Step 10: Generate & Save Presentation-Ready EDA Charts (`../outputs/eda/`)
Generates 5 clean, high-resolution charts for **Slide 6 (Dataset Statistics & Exploration)**:
1. `01_split_verdict_distribution.png` — Call-level class balance across all 4 dataset splits.
2. `02_domain_fraud_breakdown.png` — Call counts across all 10 fraud domains (Legitimate vs. Fraud).
3. `03_call_duration_distribution.png` — Distribution of call durations (seconds) by verdict.
4. `04_audio_emotion_pace_comparison.png` — Top dominant emotions and speech pace (%) comparing verdicts.
5. `05_case_type_breakdown.png` — Distribution of scenario `case_type` categories in the training split.

In [ ]:
# Chart 1: Call-Level Verdict Distribution Across Splits
fig, ax = plt.subplots(figsize=(8.5, 4.5))
plot_df = verdict_table.drop(columns=["Total"]).T
plot_df.plot(kind="bar", ax=ax, width=0.65, colormap="Set2", edgecolor="black", linewidth=0.5)
ax.set_title("ICFD-31k Call-Level Distribution Across Dataset Splits", fontsize=12, fontweight="bold")
ax.set_xlabel("Dataset Split")
ax.set_ylabel("Number of Conversations")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Final Verdict")
for container in ax.containers:
    ax.bar_label(container, fmt="{:,.0f}", fontsize=8, padding=2)
plt.tight_layout()
fig.savefig(eda_out_dir / "01_split_verdict_distribution.png", dpi=200)
plt.show()

# Chart 2: Domain Breakdown (Train — Call Level)
fig, ax = plt.subplots(figsize=(9.5, 5.5))
dom_cols = [c for c in domain_ct.columns if c not in ("Total", "Fraud_%")]
domain_plot = domain_ct[dom_cols].loc[domain_ct.sort_values("Total").index]
domain_plot.plot(kind="barh", stacked=True, ax=ax, colormap="Set2", edgecolor="black", linewidth=0.4)
ax.set_title("Training Conversations by Fraud Domain (10 Main Domains)", fontsize=12, fontweight="bold")
ax.set_xlabel("Number of Conversations")
ax.set_ylabel("Domain")
ax.legend(title="Final Verdict", loc="lower right")
plt.tight_layout()
fig.savefig(eda_out_dir / "02_domain_fraud_breakdown.png", dpi=200)
plt.show()

# Chart 3: Call Duration Distribution (Train — Seconds)
fig, ax = plt.subplots(figsize=(8.5, 4.5))
for verdict in sorted(train_calls_df["final_verdict"].dropna().unique()):
    subset = train_calls_df.loc[train_calls_df["final_verdict"] == verdict, "last_ts"]
    ax.hist(
        subset,
        bins=30,
        alpha=0.55,
        label=f"Verdict: {verdict} (median={subset.median():.0f}s)",
        edgecolor="black",
        linewidth=0.4,
    )
ax.set_title("Distribution of Call Durations in Training Split", fontsize=12, fontweight="bold")
ax.set_xlabel("Final Chunk Timestamp (Seconds)")
ax.set_ylabel("Number of Conversations")
ax.legend()
plt.tight_layout()
fig.savefig(eda_out_dir / "03_call_duration_distribution.png", dpi=200)
plt.show()

# Chart 4: Audio Emotion & Pace Comparison (Train Chunks)
if "dominant_emotion" in audio_df.columns and not pace_by_verdict.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

    top_emotions = audio_df["dominant_emotion"].value_counts().head(8).index
    emo_ct = (
        pd.crosstab(audio_df["dominant_emotion"], audio_df["final_verdict"], normalize="columns")
        .reindex(top_emotions)
        * 100
    )
    emo_ct.plot(kind="barh", ax=axes[0], colormap="Set2", edgecolor="black", linewidth=0.4)
    axes[0].invert_yaxis()
    axes[0].set_title("Top 8 Dominant Emotions (% of Chunks)", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("Percentage (%)")
    axes[0].set_ylabel("Dominant Emotion")
    axes[0].legend(title="Verdict")

    pace_by_verdict.plot(kind="bar", ax=axes[1], colormap="Set2", edgecolor="black", linewidth=0.4)
    axes[1].set_title("Speech Pace Distribution (% of Chunks)", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("Pace Category")
    axes[1].set_ylabel("Percentage (%)")
    axes[1].tick_params(axis="x", rotation=0)
    axes[1].legend(title="Verdict")

    plt.tight_layout()
    fig.savefig(eda_out_dir / "04_audio_emotion_pace_comparison.png", dpi=200)
    plt.show()

# Chart 5: Case Type Breakdown (Train — Call Level)
fig, ax = plt.subplots(figsize=(9, 4.5))
ct_plot = case_type_ct.drop(index="All", errors="ignore").drop(columns="All", errors="ignore")
ct_plot = ct_plot.loc[ct_plot.sum(axis=1).sort_values().index]
ct_plot.plot(kind="barh", stacked=True, ax=ax, colormap="Set2", edgecolor="black", linewidth=0.4)
ax.set_title("Training Conversations by Scenario Case Type", fontsize=12, fontweight="bold")
ax.set_xlabel("Number of Conversations")
ax.set_ylabel("Case Type")
ax.legend(title="Final Verdict", loc="lower right")
plt.tight_layout()
fig.savefig(eda_out_dir / "05_case_type_breakdown.png", dpi=200)
plt.show()

print("Saved all EDA charts to:", eda_out_dir.resolve())

## Step 11: Export Clean, Stitched Splits & EDA Summary (`../data/processed/` & `../outputs/eda/`)
To make **Notebook 02 (`02_preprocessing_and_features.ipynb`)** fast and memory-efficient, we:
1. Keep only the columns needed for grouping, modeling, and evaluation (dropping heavy unused JSON blobs like `cumulative_transcript_json`, `slow_thinking_rationale`, and `scenario_info_json`—cutting RAM by >50%).
2. Write each stitched, chronologically sorted split to `../data/processed/{split}_stitched.parquet` using `pyarrow.parquet.write_table()` directly (avoiding the Python 3.14 `df.to_parquet()` bug).
3. Save `eda_summary_stats.json` and `train_call_summary.csv` to `../outputs/eda/` for quick reference when filling Slide 6.

In [ ]:
# Columns kept for Notebook 02:
# - Grouping/Eval ONLY (never fed to models): conversation_uid, domain, case_type, final_verdict
# - Model Feature Inputs: chunk_timestamp, cumulative_text, cumulative_audio_analysis_json
EXPORT_COLS = [
    "conversation_uid",
    "domain",
    "case_type",
    "final_verdict",
    "chunk_timestamp",
    "cumulative_text",
    "cumulative_audio_analysis_json",
]

print(f"{'Split':<14} | {'Raw RAM (MB)':>12} | {'Slim RAM (MB)':>13} | {'Saved File (MB)':>15}")
print("-" * 64)
for name, (df, _) in splits.items():
    raw_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    slim_df = df[EXPORT_COLS].copy()
    slim_mb = slim_df.memory_usage(deep=True).sum() / (1024 ** 2)

    out_path = processed_dir / f"{name}_stitched.parquet"
    # Use PyArrow directly to avoid Python 3.14 pandas.to_parquet() ArrowKeyError
    pq.write_table(pa.Table.from_pandas(slim_df, preserve_index=False), out_path)
    file_mb = out_path.stat().st_size / (1024 ** 2)
    print(f"{name:<14} | {raw_mb:>12.1f} | {slim_mb:>13.1f} | {file_mb:>15.2f}")

# Save call-level summary CSV and JSON stats for Slide 6
train_calls_df.to_csv(eda_out_dir / "train_call_summary.csv", index=False)

eda_summary = {
    "dataset": release_summary.get("dataset", "ICFD-31k"),
    "version": release_summary.get("version", "v1.0"),
    "total_conversations": int(total_calls),
    "total_streaming_chunks": int(total_chunks),
    "zero_cross_split_overlap": not overlap_found,
    "train_repeated_consecutive_text_chunks": repeated_chunks,
    "train_repeated_consecutive_text_pct": round(repeated_chunks / len(train_df) * 100, 2),
    "call_level_verdicts_by_split": verdict_table.to_dict(),
    "leakage_columns_to_drop": LEAKAGE_COLS,
    "safe_model_input_columns": SAFE_MODEL_INPUT_COLS,
}

with open(eda_out_dir / "eda_summary_stats.json", "w", encoding="utf-8") as f:
    json.dump(eda_summary, f, indent=2)

print("\nExported stitched splits to :", processed_dir.resolve())
print("Exported Slide 6 stats to   :", (eda_out_dir / "eda_summary_stats.json").resolve())

## Ready for Notebook 02 (`02_preprocessing_and_features.ipynb`)
In the next notebook, we will load `../data/processed/{split}_stitched.parquet` directly (in ~1 second) and:
1. Drop all leakage/evaluation columns (`conversation_uid`, `domain`, `case_type`, `final_verdict`) from `X` while keeping them aligned for session-level evaluation.
2. Filter `train` to **normal/legitimate calls** for unsupervised Isolation Forest and Autoencoder training.
3. Build the numerical feature matrix: **Sentence Embeddings** (`cumulative_text`) + **Encoded Emotion & Pace** (`cumulative_audio_analysis_json`) + **Normalized Time** (`chunk_timestamp`).